# 🔍 OCR Metadata Audit Pipeline
**End-to-end image OCR, metadata extraction, structured field parsing, and quality scoring.**

Stages: `Ingestion → OCR → Fields → Layout → Semantics → Uncertainty → Scoring → Reports`


## 0 · Install & Imports

In [ ]:
# Run once
import sys
!{sys.executable} -m pip install pillow pytesseract opencv-python-headless \
    exifread anthropic pandas openpyxl rich python-magic -q
# System: sudo apt-get install -y tesseract-ocr libmagic1


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import base64, csv, hashlib, json, math, os, re, statistics, struct, textwrap, unicodedata
from dataclasses import dataclass, field, asdict
from datetime import datetime
from io import BytesIO
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# ── Image processing ──────────────────────────────────────────────────────────
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps
import exifread

# ── OCR ───────────────────────────────────────────────────────────────────────
import pytesseract
from pytesseract import Output

# ── AI ────────────────────────────────────────────────────────────────────────
import anthropic

# ── Output ────────────────────────────────────────────────────────────────────
import pandas as pd
from IPython.display import display, HTML, Markdown

print("All imports OK")


✅ All imports OK


## 1 · Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("output"); OUTPUT_DIR.mkdir(exist_ok=True)

# ── API ───────────────────────────────────────────────────────────────────────

ANTHROPIC_MODEL      = "claude-sonnet-4-20250514"
ANTHROPIC_MAX_TOKENS = 4096
ANTHROPIC_TEMP       = 0.1

# ── OCR ───────────────────────────────────────────────────────────────────────
TESS_CONFIG_AUTO   = "--oem 3 --psm 3"
TESS_CONFIG_SPARSE = "--oem 3 --psm 11"
TESS_LANG          = "eng"
CONF_LOW           = 60   # token confidence threshold

# ── Scoring weights (must sum to 1.0) ─────────────────────────────────────────
WEIGHTS = {
    "ocr_confidence": 0.20, "metadata_confidence": 0.07,
    "completeness":   0.15, "legibility":          0.12,
    "layout":         0.08, "field_accuracy":      0.15,
    "ambiguity_risk": 0.08, "hallucination_risk":  0.07,
    "reproducibility":0.08,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, "Weights must sum to 1.0"

# ── Trust thresholds ──────────────────────────────────────────────────────────
TRUST = {85: "High trust", 65: "Moderate trust", 45: "Low trust"}

# ── Regex field patterns ──────────────────────────────────────────────────────
PATTERNS = {
    "dates":    [re.compile(r'\b(\d{1,2}(?:st|nd|rd|th)?\s+\w+\s+\d{4})\b', re.I),
                 re.compile(r'\b(\w+\s+\d{1,2}(?:st|nd|rd|th)?,?\s+\d{4})\b', re.I),
                 re.compile(r'\b(\d{4}-\d{2}-\d{2})\b')],
    "emails":   [re.compile(r'\b([A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,})\b')],
    "phones":   [re.compile(r'\b(\+?\d[\d\s\-().]{7,}\d)\b')],
    "prices":   [re.compile(r'[£$€¥]\s*\d[\d,.]*')],
    "urls":     [re.compile(r'https?://\S+', re.I), re.compile(r'www\.\S+', re.I)],
    "refs":     [re.compile(r'\b([A-Z]{2,}\d{3,}[\w\-]*)\b')],
    "addresses":[re.compile(r'\b\d+[,\s]+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*(?:,\s*[A-Z][a-z]+)*\b')],
}

SENSITIVITY_KW = ["confidential","private","restricted","passport","national insurance",
                  "social security","credit card","medical","classified"]

REGION_LABELS = {
    (0,0):"top-left",(0,1):"top-centre",(0,2):"top-right",
    (1,0):"centre-left",(1,1):"centre",(1,2):"centre-right",
    (2,0):"bottom-left",(2,1):"bottom-centre",(2,2):"bottom-right",
}

print("Config loaded")


✅ Config loaded


## 2 · Data Models

In [ ]:
@dataclass
class TechMeta:
    filename:str; filepath:str; size_bytes:int; size_human:str
    ext:str; mime:str; width:int; height:int; aspect:str
    mode:str; bit_depth:Optional[int]; dpi_x:int; dpi_y:int
    width_mm:Optional[float]; height_mm:Optional[float]
    orientation:str; is_grey:bool; colours:List[str]
    exif:Dict; gps:Optional[Dict]; camera:Optional[str]
    software:Optional[str]; created:Optional[str]; modified:Optional[str]
    icc:Optional[str]; has_metadata:bool

@dataclass
class OCRToken:
    text:str; conf:int; left:int; top:int; width:int; height:int
    block:int; line:int

@dataclass
class OCRResult:
    raw:str; clean:str; tokens:List[OCRToken]
    conf_stats:Dict; words:int; chars:int; lines:int
    language:str; preprocessing:List[str]

@dataclass
class Fields:
    title:Optional[str]; subtitle:Optional[str]
    headings:List[str]; names:List[str]; orgs:List[str]
    addresses:List[str]; phones:List[str]; emails:List[str]
    urls:List[str]; dates:List[str]; times:List[str]
    prices:List[str]; refs:List[str]; warnings:List[str]
    captions:List[str]; languages:List[str]
    extra:Dict; ai_raw:Dict

@dataclass
class Region:
    label:str; row:int; col:int
    text:str; words:int; conf:int
    objects:List[str]; certainty:str

@dataclass
class Semantic:
    image_class:str; doc_type:str; subjects:List[str]
    objects:List[str]; brands:List[str]; languages:List[str]
    text_style:str; sensitivity:Dict; auth_notes:str
    observations:List[str]; is_official:bool; is_commercial:bool
    is_historical:bool; has_tables:bool; has_charts:bool
    has_signatures:bool; has_stamps:bool; has_barcodes:bool
    has_handwriting:bool

@dataclass
class UncertainItem:
    idx:int; field:str; location:str
    best:str; alts:List[str]; conf:str; reason:str

@dataclass
class Uncertainty:
    items:List[UncertainItem]; total:int; level:str

@dataclass
class Scores:
    ocr_confidence:int; metadata_confidence:int; completeness:int
    legibility:int; layout:int; field_accuracy:int
    ambiguity_risk:int; hallucination_risk:int; reproducibility:int
    overall:int; trust:str; needs_review:bool
    verdict:str; limitations:List[str]; rationale:Dict[str,str]

@dataclass
class Report:
    path:str; timestamp:str; overview:str
    tech:TechMeta; ocr:OCRResult; fields:Fields
    regions:List[Region]; semantic:Semantic
    uncertainty:Uncertainty; scores:Scores

    def to_dict(self):
        def _c(o):
            if hasattr(o,'__dataclass_fields__'): return {k:_c(v) for k,v in asdict(o).items()}
            if isinstance(o,list): return [_c(i) for i in o]
            if isinstance(o,dict): return {k:_c(v) for k,v in o.items()}
            return o
        return _c(self)

print("Data models defined")


✅ Data models defined


## 3 · Image Utilities

In [ ]:
# ── Converters ────────────────────────────────────────────────────────────────
def pil2cv(img): return cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
def cv2pil(a):   return Image.fromarray(a) if len(a.shape)==2 else Image.fromarray(cv2.cvtColor(a,cv2.COLOR_BGR2RGB))

# ── DPI & physical size ───────────────────────────────────────────────────────
def get_dpi(img):
    d = img.info.get("dpi", img.info.get("jfif_density",(72,72)))
    try: return max(int(round(d[0])),1), max(int(round(d[1])),1)
    except: return 72,72

def phys_size_mm(img):
    x,y = get_dpi(img)
    if x<=72 and y<=72: return None,None
    return round(img.width/x*25.4,1), round(img.height/y*25.4,1)

# ── Preprocessing chain ───────────────────────────────────────────────────────
def preprocess(img, target_dpi=300):
    steps=[]
    # Upscale if below target DPI
    x,_ = get_dpi(img)
    if x < target_dpi:
        s = target_dpi/x
        img = img.resize((int(img.width*s),int(img.height*s)), Image.LANCZOS); steps.append("upscale")
    # Contrast
    img = ImageEnhance.Contrast(img).enhance(1.8); steps.append("contrast")
    # OpenCV pipeline
    a = pil2cv(img)
    a = cv2.fastNlMeansDenoisingColored(a, h=10, hColor=10); steps.append("denoise")
    # Deskew via Hough lines
    grey = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(grey,50,150,apertureSize=3)
    lines = cv2.HoughLinesP(edges,1,math.pi/180,100,minLineLength=100,maxLineGap=10)
    if lines is not None:
        angles=[math.degrees(math.atan2(l[0][3]-l[0][1],l[0][2]-l[0][0])) for l in lines if l[0][2]!=l[0][0]]
        if angles:
            ang=np.median(angles)
            if 0.3<abs(ang)<15:
                h,w=a.shape[:2]; M=cv2.getRotationMatrix2D((w/2,h/2),ang,1.0)
                a=cv2.warpAffine(a,M,(w,h),flags=cv2.INTER_CUBIC,borderMode=cv2.BORDER_REPLICATE); steps.append("deskew")
    # Adaptive binarise
    grey2=cv2.cvtColor(a,cv2.COLOR_BGR2GRAY)
    a=cv2.adaptiveThreshold(grey2,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,11); steps.append("binarise")
    return cv2pil(a), steps

# ── Misc helpers ──────────────────────────────────────────────────────────────
def aspect_ratio(w,h): g=math.gcd(w,h); return f"{w//g}:{h//g}"
def human_size(n):
    for u in ("B","KB","MB","GB"):
        if n<1024: return f"{n:.1f} {u}"
        n/=1024
    return f"{n:.1f} TB"
def orientation(w,h): return "Portrait" if h>w else ("Square" if h==w else "Landscape")
def is_grey(img,tol=10):
    if img.mode=="L": return True
    a=np.array(img.convert("RGB").resize((64,64),Image.NEAREST))
    return bool(np.max(np.abs(a[:,:,0].astype(int)-a[:,:,1].astype(int)))<=tol)

def dom_colours(img,n=5):
    try:
        a=np.float32(np.array(img.convert("RGB").resize((100,100),Image.NEAREST)).reshape(-1,3))
        _,labels,centres=cv2.kmeans(a,n,None,(cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,10,1.0),10,cv2.KMEANS_RANDOM_CENTERS)
        order=np.argsort(-np.bincount(labels.flatten()))
        return ["#{:02X}{:02X}{:02X}".format(int(c[0]),int(c[1]),int(c[2])) for c in centres[order]]
    except: return []

def crop_regions(img,rows=3,cols=3):
    w,h=img.size; cw,ch=w//cols,h//rows; crops={}
    for r in range(rows):
        for c in range(cols):
            crops[(r,c)]=img.crop((c*cw,r*ch,(c+1)*cw if c<cols-1 else w,(r+1)*ch if r<rows-1 else h))
    return crops

def img_to_b64(img,max_px=2048):
    i=img.copy().convert("RGB")
    if max(i.size)>max_px:
        s=max_px/max(i.size); i=i.resize((int(i.width*s),int(i.height*s)),Image.LANCZOS)
    buf=BytesIO(); i.save(buf,format="JPEG",quality=95)
    return base64.standard_b64encode(buf.getvalue()).decode()

print("Image utilities defined")


✅ Image utilities defined


## 4 · Text Utilities

In [ ]:
def clean_text(raw):
    text="".join(c for c in raw if unicodedata.category(c)[0]!="C" or c in "\n\t\r")
    text=unicodedata.normalize("NFC",text)
    lines=[re.sub(r"[ \t]+"," ",l).strip() for l in text.splitlines()]
    out,blanks=[],0
    for l in lines:
        if l=="":
            blanks+=1
            if blanks<=1: out.append(l)
        else: blanks=0; out.append(l)
    return "\n".join(out)

def regex_fields(text):
    out={}
    for name,pats in PATTERNS.items():
        found=set()
        for p in pats:
            for m in p.finditer(text): found.add(m.group(0).strip())
        out[name]=sorted(found)
    return out

def extract_names(text):
    hon=r"(?:Mr|Mrs|Ms|Miss|Dr|Prof|Professor|Sir|Lady|Lord|Mons)\."
    pat=re.compile(rf"(?:{hon}\s+)?(?:[A-Z][a-z]+(?:[\s\-][A-Z][a-z]+){{1,3}})")
    return [n for n in dict.fromkeys(m.group(0).strip() for m in pat.finditer(text)) if len(n)>3]

def extract_orgs(text):
    stops={"AND","FOR","THE","WITH","ONLY","FROM","WILL","HEAR","COME","LAST","SEE",
           "MONDAY","TUESDAY","WEDNESDAY","THURSDAY","FRIDAY","SATURDAY","SUNDAY",
           "APRIL","JANUARY","FEBRUARY","MARCH","MAY","JUNE","JULY","AUGUST",
           "SEPTEMBER","OCTOBER","NOVEMBER","DECEMBER","EASTER","WEEK"}
    pat=re.compile(r'\b([A-Z]{3,}(?:[\s&\-][A-Z]{2,})*)\b')
    return [o for o in dict.fromkeys(m.group(0) for m in pat.finditer(text)) if o not in stops]

def sensitivity(text):
    hits=[k for k in SENSITIVITY_KW if k in text.lower()]
    strong={"passport","national insurance","social security","credit card"}
    lvl="high" if any(h in strong for h in hits) else ("moderate" if len(hits)>1 else ("low" if hits else "none"))
    return {"flag":bool(hits),"keywords":hits,"level":lvl}

def detect_lang(text):
    ENG={"the","and","of","to","in","is","it","for","on","are","with","as","at","be",
         "by","an","or","not","this","from","was","but","have","been","will","their","only"}
    return "English (probable)" if len(set(re.findall(r'\b[a-z]+\b',text.lower()))&ENG)>=3 else "Unknown"

def conf_stats(vals):
    if not vals: return {"count":0,"mean":None,"median":None,"low_pct":None,"high_pct":None}
    n=len(vals)
    return {"count":n,"mean":round(statistics.mean(vals),1),"median":statistics.median(vals),
            "low_count":sum(1 for v in vals if v<60),"high_count":sum(1 for v in vals if v>=80),
            "low_pct":round(sum(1 for v in vals if v<60)/n*100,1),
            "high_pct":round(sum(1 for v in vals if v>=80)/n*100,1)}

print("Text utilities defined")


✅ Text utilities defined


## 5 · Stage 1 — Ingestion

In [ ]:
SUPPORTED = {".jpg",".jpeg",".png",".tiff",".tif",".bmp",".webp",".gif"}
_BIT={"1":1,"L":8,"P":8,"RGB":24,"RGBA":32,"CMYK":32,"YCbCr":24,"I":32,"F":32}

def _parse_exif(path):
    try:
        with open(path,"rb") as f:
            tags=exifread.process_file(f,details=False,strict=False)
        return {k:str(v) for k,v in tags.items()}
    except: return {}

def _parse_gps(exif):
    gps={k.replace("GPS ","").strip():v for k,v in exif.items() if k.lower().startswith("gps ")}
    return gps or None

def _get(exif,*keys):
    for k in keys:
        v=exif.get(k) or exif.get(f"Image {k}") or exif.get(f"EXIF {k}")
        if v: return str(v)
    return None

def _icc(img):
    try:
        raw=img.info.get("icc_profile")
        if raw and len(raw)>40:
            off=struct.unpack(">I",raw[36:40])[0]
            if off and off+12<len(raw):
                n=struct.unpack(">I",raw[off+8:off+12])[0]
                return raw[off+12:off+12+n].decode("utf-8",errors="replace").strip("\x00")
    except: pass
    return None

def ingest(path):
    path=Path(path).resolve()
    if not path.exists(): raise FileNotFoundError(path)
    if path.suffix.lower() not in SUPPORTED: raise ValueError(f"Unsupported: {path.suffix}")
    sz=path.stat().st_size
    try: mime=__import__("magic").from_file(str(path),mime=True)
    except: mime={"jpg":"image/jpeg","jpeg":"image/jpeg","png":"image/png"}.get(path.suffix.lower().strip("."),"image/octet-stream")
    exif=_parse_exif(path)
    img=Image.open(path); img.load()
    w,h=img.size; dx,dy=get_dpi(img)
    wm,hm=phys_size_mm(img)
    return img, TechMeta(
        filename=path.name, filepath=str(path), size_bytes=sz, size_human=human_size(sz),
        ext=path.suffix.lower(), mime=mime, width=w, height=h, aspect=aspect_ratio(w,h),
        mode=img.mode, bit_depth=_BIT.get(img.mode), dpi_x=dx, dpi_y=dy,
        width_mm=wm, height_mm=hm, orientation=orientation(w,h),
        is_grey=is_grey(img), colours=dom_colours(img),
        exif=exif, gps=_parse_gps(exif), camera=_get(exif,"Make","Model"),
        software=_get(exif,"Software"),
        created=_get(exif,"DateTimeOriginal","DateTimeDigitized","DateTime"),
        modified=_get(exif,"DateTime","ModifyDate"),
        icc=_icc(img), has_metadata=bool(exif)
    )

print("Stage 1 (Ingestion) defined")


✅ Stage 1 (Ingestion) defined


## 6 · Stage 2 — OCR

In [ ]:
def _tess_tokens(img, cfg):
    try:
        d=pytesseract.image_to_data(img,lang=TESS_LANG,config=cfg,output_type=Output.DICT)
        return [OCRToken(t,int(c),int(d["left"][i]),int(d["top"][i]),
                         int(d["width"][i]),int(d["height"][i]),
                         int(d["block_num"][i]),int(d["line_num"][i]))
                for i,(t,c) in enumerate(zip(d["text"],d["conf"]))
                if t.strip() and int(c)>=0]
    except: return []

def _tess_text(img, cfg):
    try: return pytesseract.image_to_string(img,lang=TESS_LANG,config=cfg)
    except: return ""

def _merge(p,s):
    seen={l.strip() for l in p.splitlines() if l.strip()}
    extra=[l for l in s.splitlines() if l.strip() and l.strip() not in seen]
    return p+("\n\n[sparse pass]\n"+"\n".join(extra) if extra else "")

def run_ocr(img, do_preprocess=True, dual=True):
    steps=[]
    if do_preprocess:
        proc,steps=preprocess(img)
    else:
        proc=img

    raw=_tess_text(proc,TESS_CONFIG_AUTO)
    tokens=_tess_tokens(proc,TESS_CONFIG_AUTO)

    if dual:
        raw2=_tess_text(proc,TESS_CONFIG_SPARSE)
        tokens2=_tess_tokens(proc,TESS_CONFIG_SPARSE)
        existing={(t.left,t.top) for t in tokens}
        tokens+=[t for t in tokens2 if (t.left,t.top) not in existing]
        raw=_merge(raw,raw2)

    cleaned=clean_text(raw)
    confs=[t.conf for t in tokens if t.conf>=0]
    return OCRResult(
        raw=raw, clean=cleaned, tokens=tokens,
        conf_stats=conf_stats(confs),
        words=len([t for t in tokens if t.text.strip()]),
        chars=len(cleaned.replace("\n","")),
        lines=len([l for l in cleaned.splitlines() if l.strip()]),
        language=detect_lang(cleaned), preprocessing=steps
    )

print("Stage 2 (OCR) defined")


✅ Stage 2 (OCR) defined


## 7 · Stage 3 — Field Extraction

In [ ]:
_FIELD_PROMPT = """
Analyse this image and return ONLY valid JSON (no fences) with this schema:
{
  "overview": "<one paragraph>",
  "title": null, "subtitle": null,
  "headings": [], "names": [], "organisations": [],
  "addresses": [], "phone_numbers": [], "emails": [],
  "urls": [], "dates": [], "times": [], "prices": [],
  "reference_numbers": [], "warnings": [], "captions": [],
  "body_text_blocks": [], "printer_imprints": [],
  "slogans_or_calls_to_action": [],
  "roles_and_titles": [{"name":"","role":""}],
  "performers_or_acts": [{"name":"","descriptor":"","duration":"","description":""}],
  "document_class": "", "document_date": null,
  "period_start": null, "period_end": null, "creator": null,
  "language": "English", "text_style": "machine-printed",
  "contains_handwriting": false, "contains_tables": false,
  "contains_charts": false, "contains_stamps": false,
  "contains_barcodes": false, "contains_signatures": false,
  "is_historical": false, "is_official": false, "is_commercial": false,
  "sensitivity_level": "none",
  "uncertainty_items": [{"field":"","location":"","best_reading":"","alternatives":[],"confidence":"","reason":""}]
}
"""

def _ai_call(img, prompt, model=ANTHROPIC_MODEL, max_tokens=ANTHROPIC_MAX_TOKENS):
    try:
        b64=img_to_b64(img)
        client=anthropic.Anthropic()
        msg=client.messages.create(model=model,max_tokens=max_tokens,temperature=ANTHROPIC_TEMP,
            messages=[{"role":"user","content":[
                {"type":"image","source":{"type":"base64","media_type":"image/jpeg","data":b64}},
                {"type":"text","text":prompt}]}])
        raw=msg.content[0].text.strip()
        if raw.startswith("```"): raw=raw.split("```")[1]; raw=raw[4:] if raw.startswith("json") else raw
        return json.loads(raw)
    except Exception as e:
        print(f"  ⚠️  AI call failed: {e}")
        return {}

def _dedup(*lists):
    seen=set(); out=[]
    for lst in lists:
        for v in (lst or []):
            k=str(v).lower().strip()
            if k and k not in seen: seen.add(k); out.append(str(v))
    return out

def _ls(v): return [str(x) for x in v if x] if isinstance(v,list) else ([str(v)] if v else [])
def _ss(v): return v.strip() if isinstance(v,str) and v.strip() else None

def extract_fields(img, ocr):
    print("  Running regex extraction...")
    rf=regex_fields(ocr.clean)
    names=extract_names(ocr.clean)
    orgs=extract_orgs(ocr.clean)

    print("  Calling Claude Vision for field extraction...")
    ai=_ai_call(img,_FIELD_PROMPT)

    return Fields(
        title=_ss(ai.get("title")), subtitle=_ss(ai.get("subtitle")),
        headings=_ls(ai.get("headings")),
        names=_dedup(_ls(ai.get("names")),names),
        orgs=_dedup(_ls(ai.get("organisations")),orgs),
        addresses=_dedup(_ls(ai.get("addresses")),rf["addresses"]),
        phones=_dedup(_ls(ai.get("phone_numbers")),rf["phones"]),
        emails=_dedup(_ls(ai.get("emails")),rf["emails"]),
        urls=_dedup(_ls(ai.get("urls")),rf["urls"]),
        dates=_dedup(_ls(ai.get("dates")),rf["dates"]),
        times=_ls(ai.get("times")),
        prices=_dedup(_ls(ai.get("prices")),rf["prices"]),
        refs=_dedup(_ls(ai.get("reference_numbers")),rf["refs"]),
        warnings=_ls(ai.get("warnings")), captions=_ls(ai.get("captions")),
        languages=[detect_lang(ocr.clean)],
        extra={
            "overview":         _ss(ai.get("overview")),
            "body_text":        _ls(ai.get("body_text_blocks")),
            "printer":          _ls(ai.get("printer_imprints")),
            "ctas":             _ls(ai.get("slogans_or_calls_to_action")),
            "roles":            ai.get("roles_and_titles",[]),
            "performers":       ai.get("performers_or_acts",[]),
            "doc_class":        _ss(ai.get("document_class")),
            "doc_date":         _ss(ai.get("document_date")),
            "period_start":     _ss(ai.get("period_start")),
            "period_end":       _ss(ai.get("period_end")),
            "creator":          _ss(ai.get("creator")),
            "sensitivity":      sensitivity(ocr.clean),
        },
        ai_raw=ai
    )

print(" Stage 3 (Fields) defined")


✅ Stage 3 (Fields) defined


## 8 · Stage 4 — Layout Analysis

In [ ]:
_NOTES={
    "top-left":"Logo/branding/proprietor credits",
    "top-centre":"Headline title or most prominent element",
    "top-right":"Management credits or secondary headers",
    "centre-left":"Secondary billing or descriptive text",
    "centre":"Main content — most information-dense",
    "centre-right":"Supplementary information",
    "bottom-left":"Footnotes or printer imprints",
    "bottom-centre":"Calls to action or policy text",
    "bottom-right":"Reference numbers or edition info",
}

def _certainty(c): return "High" if c>=80 else ("Medium" if c>=55 else ("Low" if c>=0 else "No text"))

def _objects(txt):
    if not txt.strip(): return ["(empty)"]
    caps=sum(1 for c in txt if c.isupper())/max(len(txt),1)
    obj=[]
    if caps>0.6 and len(txt.split())<10: obj.append("display/headline text")
    if any(x in txt.upper() for x in ["MONDAY","TUESDAY","APRIL","MARCH"]): obj.append("date info")
    if any(x in txt.upper() for x in ["PRINTER","PUBLISHER","LANE"]): obj.append("imprint")
    if any(x in txt.upper() for x in ["MR.","MRS.","MISS","PROF.","MONS."]): obj.append("named person")
    if "&" in txt: obj.append("act/partnership billing")
    return obj or ["body text"]

def analyse_layout(img):
    crops=crop_regions(img)
    regions=[]
    for (r,c),crop in sorted(crops.items()):
        label=REGION_LABELS.get((r,c),f"r{r}c{c}")
        try:
            d=pytesseract.image_to_data(crop,lang=TESS_LANG,config=TESS_CONFIG_AUTO,output_type=Output.DICT)
            wc=[int(x) for t,x in zip(d["text"],d["conf"]) if t.strip() and int(x)>=0]
            txt=pytesseract.image_to_string(crop,lang=TESS_LANG,config=TESS_CONFIG_AUTO)
            txt=clean_text(txt); mc=int(sum(wc)/len(wc)) if wc else -1
        except: txt=""; mc=-1
        regions.append(Region(label=label,row=r,col=c,
            text=txt,words=len(txt.split()),conf=mc,
            objects=_objects(txt),certainty=_certainty(mc)))
    return regions

print(" Stage 4 (Layout) defined")


✅ Stage 4 (Layout) defined


## 9 · Stage 5 — Semantic Metadata

In [ ]:
_SEM_PROMPT = """
Characterise this image at a semantic level. Return ONLY valid JSON:
{
  "image_class": "scan|photo|screenshot|poster|document|other",
  "document_type": "<specific type>",
  "dominant_subjects": [], "objects_present": [],
  "logos_brands": [], "languages_visible": [],
  "text_style": "machine-printed|handwritten|mixed",
  "condition": "mint|very good|good|fair|poor",
  "authenticity_notes": "",
  "notable_observations": [],
  "is_official": false, "is_commercial": false,
  "is_historical": false,
  "contains_tables": false, "contains_charts": false,
  "contains_signatures": false, "contains_stamps": false,
  "contains_barcodes": false, "contains_handwriting": false,
  "sensitivity_level": "none|low|moderate|high",
  "sensitivity_justification": ""
}
"""

def analyse_semantic(img, fields):
    print("  Calling Claude Vision for semantic analysis...")
    ai=_ai_call(img,_SEM_PROMPT)
    def _b(k): v=ai.get(k,False); return bool(v) if isinstance(v,bool) else str(v).lower()=="true"
    def _ls2(k): v=ai.get(k,[]); return [str(x) for x in v if x] if isinstance(v,list) else []

    # Merge sensitivity: take the higher of AI and text scan
    lvl_order={"none":0,"low":1,"moderate":2,"high":3}
    text_lvl=fields.extra.get("sensitivity",{}).get("level","none")
    ai_lvl=ai.get("sensitivity_level","none")
    final_lvl=max(text_lvl,ai_lvl,key=lambda x:lvl_order.get(x,0))

    return Semantic(
        image_class=ai.get("image_class","unknown"),
        doc_type=ai.get("document_type","unknown"),
        subjects=_ls2("dominant_subjects"), objects=_ls2("objects_present"),
        brands=_ls2("logos_brands"), languages=_ls2("languages_visible"),
        text_style=ai.get("text_style","unknown"),
        sensitivity={"level":final_lvl,"flag":final_lvl!="none",
                     "keywords":fields.extra.get("sensitivity",{}).get("keywords",[]),
                     "justification":ai.get("sensitivity_justification","")},
        auth_notes=ai.get("authenticity_notes",""),
        observations=_ls2("notable_observations"),
        is_official=_b("is_official"), is_commercial=_b("is_commercial"),
        is_historical=_b("is_historical"), has_tables=_b("contains_tables"),
        has_charts=_b("contains_charts"), has_signatures=_b("contains_signatures"),
        has_stamps=_b("contains_stamps"), has_barcodes=_b("contains_barcodes"),
        has_handwriting=_b("contains_handwriting")
    )

print(" Stage 5 (Semantics) defined")


✅ Stage 5 (Semantics) defined


## 10 · Stage 6 — Uncertainty Report

In [ ]:
_HEURISTIC_PATS=[
    (re.compile(r'\b[0O][0O]+\b'),      "0/O confusion"),
    (re.compile(r'\b[1lI][1lI]+\b'),    "1/l/I confusion"),
    (re.compile(r'\brn\b'),              "rn/m confusion"),
    (re.compile(r'[a-z][A-Z][a-z]'),      "mid-word capital"),
]

def build_uncertainty(ocr, fields):
    items=[]
    seen=set()

    def _add(field,loc,best,alts,conf,reason):
        k=best.lower().strip()
        if k and k not in seen:
            seen.add(k)
            items.append(UncertainItem(len(items)+1,field,loc,best,alts,conf,reason))

    # Source 1: low-confidence OCR tokens
    for t in ocr.tokens:
        if 0<=t.conf<CONF_LOW:
            _add("ocr_token",f"block {t.block} line {t.line}",t.text,[],
                 "Low",f"Tesseract confidence: {t.conf}/100")

    # Source 2: AI-flagged items
    for u in fields.ai_raw.get("uncertainty_items",[]):
        if isinstance(u,dict):
            alts=u.get("alternatives",[]); alts=[alts] if isinstance(alts,str) else alts
            _add(u.get("field",""),u.get("location",""),u.get("best_reading",""),
                 alts,u.get("confidence","Medium"),u.get("reason","AI flagged"))

    # Source 3: heuristic patterns
    for pat,label in _HEURISTIC_PATS:
        for m in pat.finditer(ocr.clean):
            ctx=ocr.clean[max(0,m.start()-15):m.end()+15].replace("\n"," ")
            _add(f"heuristic:{label}",f"offset {m.start()}",m.group(0),[],
                 "Medium",f"{label} — context: '…{ctx}…'")
            if len(items)>=30: break

    n=len(items); w=max(ocr.words,1); ratio=n/w
    lvl="Low" if ratio<0.02 and sum(1 for i in items if i.conf=="Low")<3 else         ("High" if ratio>=0.08 else "Moderate")
    return Uncertainty(items=items,total=n,level=lvl)

print(" Stage 6 (Uncertainty) defined")


✅ Stage 6 (Uncertainty) defined


## 11 · Stage 7 — Scoring

In [ ]:
def _trust(v):
    for thr in sorted(TRUST,reverse=True):
        if v>=thr: return TRUST[thr],v<65
    return "Needs human review",True

def score_all(tech, ocr, fields, regions, semantic, uncertainty):
    R={}; RAT={}
    def _s(name,score,rationale): R[name]=max(0,min(100,int(score))); RAT[name]=rationale

    # 1 OCR confidence
    m=ocr.conf_stats.get("mean") or 50; lp=ocr.conf_stats.get("low_pct") or 0
    _s("ocr_confidence", m-lp*0.5, f"Mean={m:.1f}, low_pct={lp:.1f}%")

    # 2 Metadata confidence
    ms=10+min(40,len(tech.exif)*2)+(15 if tech.created else 0)+(10 if tech.camera else 0)+(10 if tech.gps else 0)
    _s("metadata_confidence", ms, f"EXIF tags={len(tech.exif)}, created={tech.created}")

    # 3 Completeness
    non_empty=sum(1 for r in regions if r.words>0)
    filled=sum(1 for f in [fields.title,fields.dates,fields.names,fields.orgs,ocr.clean] if f)
    _s("completeness", non_empty/len(regions)*40+filled/5*60, f"{non_empty}/{len(regions)} regions, {filled}/5 fields")

    # 4 Legibility
    mp=(tech.width*tech.height)/1e6
    rs=100 if mp>=4 else (int(60+mp/4*40) if mp>=1 else 20)
    _s("legibility", rs*0.3+m*0.5+non_empty/len(regions)*100*0.2, f"{mp:.1f}MP, conf={m:.1f}")

    # 5 Layout preservation
    rc=[r.conf for r in regions if r.conf>=0]
    lmean=statistics.mean(rc) if rc else 0
    lstd=statistics.stdev(rc) if len(rc)>1 else 0
    _s("layout", lmean-min(lstd,40), f"region conf mean={lmean:.1f} std={lstd:.1f}")

    # 6 Field accuracy
    yr=re.compile(r'\b(1[0-9]{{3}}|2[0-9]{{3}})\b')
    bad=[d for d in fields.dates if not yr.search(d)]
    fs=90-len(bad)*5-(8 if not fields.title and not fields.headings else 0)
    _s("field_accuracy", fs, f"bad_dates={len(bad)}, title={'yes' if fields.title else 'no'}")

    # 7 Ambiguity risk (high score = low risk)
    ratio=uncertainty.total/max(ocr.words,1)
    base=max(0,min(100,int(100-ratio*400)))
    if uncertainty.level=="Low": base=max(base,75)
    elif uncertainty.level=="High": base=min(base,40)
    _s("ambiguity_risk", base, f"{uncertainty.total} items / {ocr.words} words = {ratio:.3f}")

    # 8 Hallucination risk (high = low risk)
    ocr_l=ocr.clean.lower()
    absent=[n for n in fields.ai_raw.get("names",[]) or []
            if n and not any(w.lower() in ocr_l for w in n.split())]
    _s("hallucination_risk", 95-min(30,len(absent)*5), f"{len(absent)} AI names absent from OCR")

    # 9 Reproducibility
    _s("reproducibility", max(0,int(m)-int(lp*0.4)-min(15,uncertainty.total)),
       f"mean={m:.1f} low_pct={lp:.1f}% uncertain={uncertainty.total}")

    # 10 Overall (weighted)
    overall=max(0,min(100,int(round(sum(R[k]*WEIGHTS[k] for k in WEIGHTS if k in R)))))
    trust,review=_trust(overall)

    weak=[k for k,v in R.items() if v<60]; strong=[k for k,v in R.items() if v>=85]
    verdict=f"Overall: {overall}/100 ({trust}). Strong: {strong}. Weak: {weak}."

    limits=[]
    if not tech.has_metadata:  limits.append("No embedded EXIF metadata.")
    if lp>20:                  limits.append(f"{lp:.1f}% low-confidence OCR tokens.")
    if uncertainty.total>10:   limits.append(f"{uncertainty.total} uncertainty items flagged.")
    if not limits:             limits.append("No significant limitations.")

    return Scores(**R,overall=overall,trust=trust,needs_review=review,
                  verdict=verdict,limitations=limits,rationale=RAT)

print(" Stage 7 (Scoring) defined")


✅ Stage 7 (Scoring) defined


## 12 · Stage 8 — Report Output

In [ ]:
def _stem(rpt): return f"{Path(rpt.path).stem}_{rpt.timestamp.replace(':','-').replace(' ','_')}"

def _bar(v,w=20):
    f=int(v/100*w); return f"[{'█'*f+'░'*(w-f)}] {v}/100"

def _col(v): return "#27ae60" if v>=80 else ("#f39c12" if v>=60 else "#e74c3c")

def write_json(rpt):
    p=OUTPUT_DIR/f"{_stem(rpt)}.json"
    p.write_text(json.dumps(rpt.to_dict(),indent=2,ensure_ascii=False,default=str)); return p

def write_csv(rpt):
    f=rpt.fields; t=rpt.tech; o=rpt.ocr; sc=rpt.scores; s=rpt.semantic; u=rpt.uncertainty
    def j(l): return "; ".join(str(x) for x in l) if l else ""
    row={"filename":t.filename,"filepath":t.filepath,"timestamp":rpt.timestamp,
         "width_px":t.width,"height_px":t.height,"orientation":t.orientation,
         "file_size":t.size_human,"mime":t.mime,"dpi_x":t.dpi_x,"dpi_y":t.dpi_y,
         "width_mm":t.width_mm or "","height_mm":t.height_mm or "","exif_tags":len(t.exif),
         "ocr_words":o.words,"ocr_lines":o.lines,"ocr_mean_conf":o.conf_stats.get("mean",""),
         "ocr_low_pct":o.conf_stats.get("low_pct",""),"language":o.language,
         "title":f.title or "","subtitle":f.subtitle or "","dates":j(f.dates),
         "names":j(f.names),"organisations":j(f.orgs),"addresses":j(f.addresses),
         "doc_class":f.extra.get("doc_class",""),"doc_date":f.extra.get("doc_date",""),
         "period_start":f.extra.get("period_start",""),"period_end":f.extra.get("period_end",""),
         "creator":f.extra.get("creator",""),"image_class":s.image_class,
         "doc_type":s.doc_type,"sensitivity":s.sensitivity.get("level",""),
         "is_historical":s.is_historical,"uncertainty_total":u.total,"ambiguity":u.level,
         "score_ocr":sc.ocr_confidence,"score_metadata":sc.metadata_confidence,
         "score_completeness":sc.completeness,"score_legibility":sc.legibility,
         "score_layout":sc.layout,"score_fields":sc.field_accuracy,
         "score_ambiguity":sc.ambiguity_risk,"score_hallucination":sc.hallucination_risk,
         "score_reproducibility":sc.reproducibility,"score_overall":sc.overall,
         "trust":sc.trust,"needs_review":sc.needs_review,
         # Catalogue columns
         "cat_title":f.title or j(f.headings[:1]),
         "cat_subjects":j(f.names),"cat_description":(f.extra.get("overview") or "")[:400],
         "cat_date_of_subject":j(f.dates[:1]),"cat_period_start":f.extra.get("period_start",""),
         "cat_period_end":f.extra.get("period_end",""),"cat_creator":f.extra.get("creator",""),
         "cat_collection_name":"","cat_date_created":f.extra.get("doc_date",""),
         "cat_length_mm":t.height_mm or "","cat_breadth_mm":t.width_mm or "",
         "cat_orientation":t.orientation}
    p=OUTPUT_DIR/f"{_stem(rpt)}.csv"
    with open(p,"w",newline="",encoding="utf-8") as fh:
        w=csv.DictWriter(fh,fieldnames=list(row.keys())); w.writeheader(); w.writerow(row)
    return p

def write_markdown(rpt):
    t=rpt.tech; o=rpt.ocr; f=rpt.fields; sc=rpt.scores; s=rpt.semantic; u=rpt.uncertainty
    lines=["# IMAGE ANALYSIS REPORT","",
           f"**File:** `{t.filename}`  **Run:** {rpt.timestamp}","","---",
           "## Overview","",rpt.overview or "","",
           "## Technical Metadata","",
           f"| Field | Value |","| --- | --- |",
           *[f"| {k} | {v} |" for k,v in [
               ("Dimensions",f"{t.width}×{t.height}px"),("File size",t.size_human),
               ("Format",t.mime),("Orientation",t.orientation),("DPI",f"{t.dpi_x}×{t.dpi_y}"),
               ("Physical size",f"{t.width_mm}×{t.height_mm}mm" if t.width_mm else "N/A"),
               ("Colour mode",t.mode),("EXIF tags",len(t.exif)),("GPS","Yes" if t.gps else "No"),
               ("Camera",t.camera or "N/A"),("Created",t.created or "N/A")]],
           "","## Raw OCR Text","","```text",o.raw,"```","",
           "## Structured Fields","",
           "| Field | Value |","| --- | --- |",
           *[f"| **{k}** | {v} |" for k,v in [
               ("Title",f.title or "—"),("Dates","; ".join(f.dates)),
               ("Names","; ".join(f.names[:8])),("Organisations","; ".join(f.orgs[:8])),
               ("Doc class",f.extra.get("doc_class","")),("Creator",f.extra.get("creator","")),
               ("Period",f"{f.extra.get('period_start','')} → {f.extra.get('period_end','')}")
           ]],
           "","## Regions","","| Region | Words | Conf | Certainty | Preview |","| --- | --- | --- | --- | --- |",
           *[f"| {r.label} | {r.words} | {r.conf} | {r.certainty} | {r.text[:50].replace(chr(10),' ')}… |"
             for r in rpt.regions],
           "","## Scores","","| Criterion | Score | Bar |","| --- | --- | --- |",
           *[f"| {k.replace('_',' ').title()} | **{getattr(sc,k)}** | `{_bar(getattr(sc,k),15)}` |"
             for k in ["ocr_confidence","metadata_confidence","completeness","legibility",
                       "layout","field_accuracy","ambiguity_risk","hallucination_risk","reproducibility"]],
           f"| **Overall** | **{sc.overall}** | `{_bar(sc.overall,15)}` |","",
           "## Verdict","",f"**Trust:** {sc.trust}  **Review:** {'Yes' if sc.needs_review else 'No'}","",
           sc.verdict,"","**Limitations:**",
           *[f"- {l}" for l in sc.limitations]]
    p=OUTPUT_DIR/f"{_stem(rpt)}.md"
    p.write_text("\n".join(lines),encoding="utf-8"); return p

def write_html(rpt):
    t=rpt.tech; o=rpt.ocr; f=rpt.fields; sc=rpt.scores; s=rpt.semantic; u=rpt.uncertainty
    def e(v): return str(v).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    def bar(v):
        return (f'<div style="display:inline-flex;align-items:center;gap:8px">'
                f'<div style="width:140px;height:10px;background:#e0e0e0;border-radius:5px">'
                f'<div style="width:{v}%;height:100%;background:{_col(v)};border-radius:5px"></div></div>'
                f'<b style="color:{_col(v)}">{v}</b></div>')

    tc={"High trust":"#27ae60","Moderate trust":"#f39c12","Low trust":"#e67e22","Needs human review":"#e74c3c"}
    score_rows="".join(f"<tr><td>{k.replace('_',' ').title()}</td><td>{bar(getattr(sc,k))}</td>"
                       f"<td style='font-size:.8em;color:#666'>{e(sc.rationale.get(k,'')[:80])}</td></tr>"
                       for k in ["ocr_confidence","metadata_confidence","completeness","legibility",
                                  "layout","field_accuracy","ambiguity_risk","hallucination_risk","reproducibility"])
    score_rows+=f"<tr style='font-weight:bold'><td>Overall Reliability</td><td>{bar(sc.overall)}</td><td></td></tr>"

    unc_rows="".join(f"<tr><td>{i.idx}</td><td>{e(i.field)}</td><td><code>{e(i.best)}</code></td>"
                     f"<td style='color:{'#e74c3c' if i.conf=='Low' else '#f39c12'}'>{i.conf}</td>"
                     f"<td style='font-size:.8em'>{e(i.reason[:80])}</td></tr>" for i in u.items[:20])

    perf=f.extra.get("performers",[]) or []
    perf_rows="".join(f"<tr><td>{e(p.get('name',''))}</td><td>{e(p.get('descriptor',''))}</td>"
                      f"<td>{e(p.get('duration',''))}</td><td style='font-size:.8em'>{e(p.get('description',''))}</td></tr>"
                      for p in perf if isinstance(p,dict))

    html=f"""<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">
<title>OCR Report — {e(t.filename)}</title>
<style>
body{{font-family:Segoe UI,Arial,sans-serif;font-size:14px;margin:32px;background:#f8f9fc;color:#222}}
h1{{color:#1a237e;border-bottom:3px solid #1a237e;padding-bottom:8px}}
h2{{color:#283593;border-bottom:1px solid #c5cae9;margin-top:28px}}
table{{border-collapse:collapse;width:100%;background:#fff;margin-bottom:16px}}
th{{background:#3949ab;color:#fff;padding:8px 12px;text-align:left;font-size:.9em}}
td{{padding:6px 12px;border-bottom:1px solid #e8eaf6;vertical-align:top}}
tr:nth-child(even) td{{background:#f3f4fa}}
pre{{background:#263238;color:#eceff1;padding:16px;border-radius:6px;overflow-x:auto;white-space:pre-wrap;font-size:.82em}}
.badge{{padding:10px 20px;border-radius:6px;color:#fff;font-weight:bold;font-size:1.05em;display:inline-block}}
.grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin:12px 0}}
.card{{background:#fff;border:1px solid #c5cae9;border-radius:6px;padding:10px}}
.cl{{font-size:.75em;color:#7986cb;text-transform:uppercase;font-weight:bold}}
</style></head><body>
<h1>📄 OCR Metadata Audit Report</h1>
<p><b>File:</b> <code>{e(t.filename)}</code> &nbsp;|&nbsp; <b>Run:</b> {e(rpt.timestamp)}</p>
<div class="badge" style="background:{tc.get(sc.trust,'#7f8c8d')}">{sc.trust} — {sc.overall}/100</div>

<h2>Technical Metadata</h2>
<div class="grid">
{''.join(f'<div class="card"><div class="cl">{k}</div><div>{v}</div></div>' for k,v in [
    ("Dimensions",f"{t.width}×{t.height}px"),("File Size",t.size_human),("Format",t.mime),
    ("Orientation",t.orientation),("DPI",f"{t.dpi_x}×{t.dpi_y}"),
    ("Physical Size",f"{t.width_mm}×{t.height_mm}mm" if t.width_mm else "N/A"),
    ("Colour Mode",t.mode),("EXIF Tags",len(t.exif)),("GPS","✅" if t.gps else "✗")])}
</div>

<h2>Raw OCR Text</h2><pre>{e(o.raw)}</pre>

<h2>Structured Fields</h2>
<table><tr><th>Field</th><th>Value</th></tr>
{''.join(f"<tr><td><b>{k}</b></td><td>{e(str(v))}</td></tr>" for k,v in [
    ("Title",f.title or "—"),("Subtitle",f.subtitle or "—"),
    ("Dates","; ".join(f.dates)),("Names","; ".join(f.names)),
    ("Organisations","; ".join(f.orgs)),("Addresses","; ".join(f.addresses)),
    ("Document class",f.extra.get("doc_class","")),("Creator",f.extra.get("creator","")),
    ("Period",f"{f.extra.get('period_start','')} → {f.extra.get('period_end','')}")])}
</table>

{'<h2>Performers / Acts</h2><table><tr><th>Name</th><th>Descriptor</th><th>Duration</th><th>Description</th></tr>'+perf_rows+'</table>' if perf_rows else ''}

<h2>Region Analysis</h2>
<table><tr><th>Region</th><th>Words</th><th>Conf</th><th>Certainty</th><th>Preview</th></tr>
{''.join(f"<tr><td><b>{r.label}</b></td><td>{r.words}</td><td>{r.conf}</td><td>{r.certainty}</td><td style='font-size:.8em'>{e(r.text[:60])}</td></tr>" for r in rpt.regions)}
</table>

<h2>Uncertainty Report ({u.total} items)</h2>
{'<table><tr><th>#</th><th>Field</th><th>Best Reading</th><th>Confidence</th><th>Reason</th></tr>'+unc_rows+'</table>' if unc_rows else '<p><i>None</i></p>'}

<h2>Evaluation Scores</h2>
<table><tr><th>Criterion</th><th>Score</th><th>Rationale</th></tr>{score_rows}</table>

<h2>Final Verdict</h2>
<div class="badge" style="background:{tc.get(sc.trust,'#7f8c8d')}">{sc.trust}</div>
<p><b>Human review:</b> {'⚠️ Yes' if sc.needs_review else '✅ No'}</p>
<p>{e(sc.verdict)}</p>
<ul>{''.join(f'<li>{e(l)}</li>' for l in sc.limitations)}</ul>
<hr><p style="font-size:.8em;color:#9e9e9e">Generated {e(rpt.timestamp)}</p>
</body></html>"""
    p=OUTPUT_DIR/f"{_stem(rpt)}.html"
    p.write_text(html,encoding="utf-8"); return p

print(" Stage 8 (Reports) defined")


✅ Stage 8 (Reports) defined


## 13 · Pipeline Orchestrator

In [ ]:
def run_pipeline(image_path, preprocess_ocr=True, dual_ocr=True,
                 formats=("json","csv","markdown","html")):
    ts=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n{'='*60}\nOCR PIPELINE  |  {Path(image_path).name}  |  {ts}\n{'='*60}")

    print("\n[1/7] Ingesting...")
    img,tech=ingest(image_path)
    print(f"  {tech.width}×{tech.height}px  {tech.size_human}  {tech.orientation}")

    print("\n[2/7] Running OCR...")
    ocr=run_ocr(img,do_preprocess=preprocess_ocr,dual=dual_ocr)
    print(f"  {ocr.words} words  {ocr.lines} lines  mean_conf={ocr.conf_stats.get('mean')}")

    print("\n[3/7] Extracting fields...")
    fields=extract_fields(img,ocr)
    print(f"  {len(fields.names)} names  {len(fields.dates)} dates  {len(fields.orgs)} orgs")

    print("\n[4/7] Analysing layout...")
    regions=analyse_layout(img)
    print(f"  {sum(1 for r in regions if r.words>0)}/9 regions have text")

    print("\n[5/7] Semantic analysis...")
    semantic=analyse_semantic(img,fields)
    print(f"  class={semantic.image_class}  type={semantic.doc_type}")

    print("\n[6/7] Uncertainty report...")
    unc=build_uncertainty(ocr,fields)
    print(f"  {unc.total} items  level={unc.level}")

    print("\n[7/7] Scoring...")
    scores=score_all(tech,ocr,fields,regions,semantic,unc)
    print(f"  Overall: {scores.overall}/100  →  {scores.trust}")

    overview=fields.extra.get("overview") or fields.ai_raw.get("overview") or ""
    report=Report(path=str(Path(image_path).resolve()),timestamp=ts,overview=overview,
                  tech=tech,ocr=ocr,fields=fields,regions=regions,semantic=semantic,
                  uncertainty=unc,scores=scores)

    print("\nWriting outputs...")
    paths={}
    if "json"     in formats: paths["json"]     = write_json(report);     print(f"  ✅ {paths['json']}")
    if "csv"      in formats: paths["csv"]      = write_csv(report);      print(f"  ✅ {paths['csv']}")
    if "markdown" in formats: paths["markdown"] = write_markdown(report); print(f"  ✅ {paths['markdown']}")
    if "html"     in formats: paths["html"]     = write_html(report);     print(f"  ✅ {paths['html']}")

    print(f"\n{'='*60}\n✅ DONE  |  {scores.overall}/100  |  {scores.trust}\n{'='*60}\n")
    return report, paths

print(" Orchestrator defined — ready to run")


✅ Orchestrator defined — ready to run


## 14 · Run the Pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive/Theatre_posters'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['NFA178R12-10.jpg', 'NFA178R12-208.jpg', 'NFA178R12-26.jpg', 'NFA178R12-18.jpg', 'NFA178R12-209.jpg', 'READ_ME.docx', 'NFA178R12-37.jpg', 'NFA178R12-17.jpg', 'NFA178R12-204.jpg', 'NFA178R12-205.jpg', 'NFA178R12-28.jpg', 'NFA178R12-267.jpg', 'NFA178R12-30.jpg', 'NFA178R12-40.jpg', 'NFA178R12-38.jpg', 'NFA178R12-104.jpg', 'NFA178R12-103.jpg', 'NFA178R12-263.jpg', 'NFA178R12-262.jpg', 'NFA178R12-207.jpg', 'NFA178R12-258.jpg', 'Argyle Theatre poster collection metadata.xlsx', 'NFA178R12-41.jpg', 'NFA178R12-39.jpg', 'NFA178R12-9.jpg', 'NFA178R12-259.jpg', 'NFA178R12-211.jpg', 'NFA178R12-35.jpg', 'NFA178R12-43.jpg', 'NFA178R12-16.jpg', 'NFA178R12-215.jpg', 'NFA178R12-220.jpg']


In [ ]:
!apt-get install -y tesseract-ocr libmagic1 -q
!pip install pillow pytesseract opencv-python-headless exifread anthropic pandas openpyxl rich python-magic -q

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
libmagic1 is already the newest version (1:5.41-3ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os


INPUT_FOLDER  = Path('/content/drive/MyDrive/Theatre_posters')
OUTPUT_FOLDER = Path('/content/drive/MyDrive/output_batch')
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# ── Find all images ────────────────────────────────────────────────────────────
SUPPORTED = {".jpg", ".jpeg", ".png", ".tiff", ".tif", ".bmp", ".webp"}
images = sorted([f for f in INPUT_FOLDER.iterdir() if f.suffix.lower() in SUPPORTED])
print(f"Found {len(images)} image(s) in {INPUT_FOLDER}")

# ── Patch OUTPUT_DIR to write into your Google Drive output folder ─────────────
 # if using the .py version
# If everything is in the notebook, just reassign the global:
OUTPUT_DIR = OUTPUT_FOLDER

# ── Batch run ──────────────────────────────────────────────────────────────────
all_reports = []
failed      = []

for i, img_path in enumerate(images, 1):
    print(f"\n{'='*60}")
    print(f"Processing {i}/{len(images)}: {img_path.name}")
    print('='*60)
    try:
        report, output_paths = run_pipeline(
            img_path,
            preprocess_ocr = True,
            dual_ocr       = False,
            formats        = ("json", "csv", "markdown", "html"),
        )
        all_reports.append(report)
        print(f"✅ Done — {report.scores.overall}/100 — {report.scores.trust}")
    except Exception as e:
        print(f"❌ FAILED: {img_path.name} — {e}")
        failed.append(img_path.name)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Batch complete: {len(all_reports)} succeeded, {len(failed)} failed")
if failed:
    print(f"Failed files: {failed}")

# ── Combined catalogue CSV (all images in one sheet) ──────────────────────────
if all_reports:
    rows = []
    for r in all_reports:
        rows.append({
            "filename":       r.tech.filename,
            "title":          r.fields.title or "",
            "subjects":       "; ".join(r.fields.names[:6]),
            "description":    (r.fields.extra.get("overview") or "")[:300],
            "date_of_subject":"; ".join(r.fields.dates[:1]),
            "period_start":   r.fields.extra.get("period_start") or "",
            "period_end":     r.fields.extra.get("period_end") or "",
            "creator":        r.fields.extra.get("creator") or "",
            "collection_name":"",
            "date_created":   r.fields.extra.get("doc_date") or "",
            "length_mm":      r.tech.height_mm or "",
            "breadth_mm":     r.tech.width_mm or "",
            "orientation":    r.tech.orientation,
            "overall_score":  r.scores.overall,
            "trust_level":    r.scores.trust,
        })
    combined_csv = OUTPUT_FOLDER / "batch_catalogue.csv"
    pd.DataFrame(rows).to_csv(combined_csv, index=False)
    print(f"\n Combined catalogue: {combined_csv}")
    display(pd.DataFrame(rows))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 30 image(s) in /content/drive/MyDrive/Theatre_posters

Processing 1/30: NFA178R12-10.jpg

OCR PIPELINE  |  NFA178R12-10.jpg  |  2026-04-01 13:29:10

[1/7] Ingesting...
  863×2500px  1.7 MB  Portrait

[2/7] Running OCR...
  279 words  52 lines  mean_conf=64.3

[3/7] Extracting fields...
  Running regex extraction...
  Calling Claude Vision for field extraction...
  ⚠️  AI call failed: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CZd8c1fuugg53XdrrjFUh'}
  29 names  0 dates  25 orgs

[4/7] Analysing layout...
  9/9 regions have text

[5/7] Semantic analysis...
  Calling Claude Vision for semantic analysis...
  ⚠️  AI call failed: Error code: 400 - {'type': 'error', 'e

,filename,title,subjects,description,date_of_subject,period_start,period_end,creator,collection_name,date_created,length_mm,breadth_mm,orientation,overall_score,trust_level
0,NFA178R12-10.jpg,,Argyle Street; Gove Blt; Tng Bogegrest; Oole-D...,,,,,,,,211.7,73.1,Portrait,56,Low trust
1,NFA178R12-103.jpg,,The Renowned Pantomimiats; Yeat Favourite; The...,,,,,,,,211.7,71.5,Portrait,50,Low trust
2,NFA178R12-104.jpg,,Important Engagement; Popular Favourites; Prem...,,,,,,,,211.7,74.6,Portrait,51,Low trust
3,NFA178R12-16.jpg,,Freeeteenenaerel Peretti; The Capi,,,,,,,,211.7,73.9,Portrait,39,Needs human review
4,NFA178R12-17.jpg,,Expensive Engagement; London Pavilion; Ae Tee ...,,,,,,,,211.7,75.7,Portrait,49,Low trust
5,NFA178R12-18.jpg,,Argyle Strest; By Spal; Gomis Vooalist; Privat...,,,,,,,,211.7,72.7,Portrait,46,Low trust
6,NFA178R12-204.jpg,,Reis Prepeteter; Mamegec Ds; Sth\nAnd Twice Ni...,,"DECEMBER 23rd, 1911",,,,,,211.7,70.5,Portrait,55,Low trust
7,NFA178R12-205.jpg,,Sele Proprietor; Clever Comedienne; New Songs;...,,,,,,,,211.7,70.1,Portrait,53,Low trust
8,NFA178R12-207.jpg,,New Songs; Firat Time; Bonar Gocdienn; Ri Se; ...,,,,,,,,211.7,70.9,Portrait,51,Low trust
9,NFA178R12-208.jpg,,Mele Frei; Renewmed Aetreas; Tih Glace; The Ma...,,,,,,,,211.7,70.7,Portrait,46,Low trust
